# 02 — Exploración de Datos (EDA) — Enfoque Ocupación

Fase **Data Understanding** de CRISP-DM. Revisamos los cuatro datasets para:

- Conocer volumen, tipos de variables y calidad (nulos, duplicados, outliers).
- Entender la **variable objetivo** `available` (disponibilidad diaria del apartamento) y las variables auxiliares (`availability_365`, `estimated_occupancy_l365d`).
- Detectar problemas a resolver en la fase de limpieza (notebook 03).

> **Pivot respecto al planteamiento inicial:** Inside Airbnb ya no publica la columna `price`, por lo que la variable objetivo del proyecto pasa de precio a **ocupación** (disponibilidad). La variable binaria es `occupied = (available == 'f')`.

No se modifican los datos. Solo se leen y se visualizan.

## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
pd.set_option('display.max_columns', 100)

RAW_DIR = Path('../data/raw')
assert RAW_DIR.exists(), 'La carpeta data/raw no existe. Ejecuta antes 01_descarga_datos.ipynb'

print('Setup OK')
print(f'Datos en: {RAW_DIR.resolve()}')

---
## 1. Listings — Apartamentos de Airbnb

Fichero principal con las características estáticas de cada alojamiento y los **indicadores agregados de ocupación** (availability_30/60/90/365, estimated_occupancy_l365d).

In [ ]:
listings = pd.read_csv(RAW_DIR / 'listings.csv')
print(f'Dimensiones: {listings.shape[0]:,} filas × {listings.shape[1]} columnas')
print(f'Memoria : {listings.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

In [ ]:
# Nulos — solo columnas con al menos un nulo
nulls = listings.isna().sum()
nulls_pct = (nulls / len(listings) * 100).round(2)
nulls_df = pd.DataFrame({'nulos': nulls, 'pct': nulls_pct})
nulls_df = nulls_df[nulls_df['nulos'] > 0].sort_values('pct', ascending=False)
print(f'Columnas con nulos: {len(nulls_df)} de {listings.shape[1]}')
nulls_df.head(15)

In [ ]:
# Subconjunto útil para predecir ocupación (sin price, está 100% nulo en los scrapes recientes)
cols_utiles = [
    'id', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    'latitude', 'longitude',
    'property_type', 'room_type',
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'maximum_nights',
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
    'estimated_occupancy_l365d',
    'number_of_reviews', 'reviews_per_month',
    'review_scores_rating', 'review_scores_location', 'review_scores_value',
    'host_is_superhost', 'instant_bookable'
]

listings_mini = listings[cols_utiles].copy()
print(f'Columnas seleccionadas: {len(cols_utiles)} de {listings.shape[1]}')
listings_mini.head()

In [ ]:
# Distribución de los indicadores de ocupación agregados
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(listings_mini['availability_365'], bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('availability_365 (días disponibles en 1 año)')
axes[0].set_xlabel('Días disponibles')

sns.histplot(listings_mini['estimated_occupancy_l365d'], bins=40, ax=axes[1], color='darkorange')
axes[1].set_title('estimated_occupancy_l365d (días ocupados estimados)')
axes[1].set_xlabel('Días ocupados')

plt.tight_layout()
plt.show()

print('\nEstadísticas:')
print(listings_mini[['availability_365','estimated_occupancy_l365d']].describe().round(1))

In [ ]:
# Ocupación estimada por tipo de habitación
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=listings_mini, x='room_type', y='estimated_occupancy_l365d',
            ax=axes[0], color='steelblue')
axes[0].set_title('Ocupación estimada por room_type')
axes[0].set_xlabel('')
axes[0].set_ylabel('Días ocupados estimados (últimos 365)')
axes[0].tick_params(axis='x', rotation=20)

top_props = listings_mini['property_type'].value_counts().head(8).index
sns.boxplot(data=listings_mini[listings_mini['property_type'].isin(top_props)],
            x='property_type', y='estimated_occupancy_l365d',
            ax=axes[1], color='steelblue')
axes[1].set_title('Ocupación estimada por property_type (top 8)')
axes[1].set_xlabel('')
axes[1].set_ylabel('Días ocupados estimados')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Ocupación mediana por barrio — top 15 más ocupados
ocup_barrio = (listings_mini
               .groupby('neighbourhood_cleansed')['estimated_occupancy_l365d']
               .agg(['median', 'count'])
               .query('count >= 30')
               .sort_values('median', ascending=False)
               .head(15))

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=ocup_barrio['median'], y=ocup_barrio.index, ax=ax, color='steelblue')
ax.set_title('Ocupación mediana anual por barrio (top 15, mínimo 30 listings)')
ax.set_xlabel('Días ocupados estimados (mediana)')
ax.set_ylabel('Barrio')
plt.tight_layout()
plt.show()

### Observaciones — Listings

- `price` está 100 % vacío — confirmado el pivot a ocupación como objetivo.
- Las variables `availability_365` y `estimated_occupancy_l365d` no tienen nulos y son buenas candidatas como **variable objetivo auxiliar** a nivel anuncio (granularidad listing, no listing × día).
- Mediana de `availability_365` = 245 días; mediana de `estimated_occupancy_l365d` = 18 días → los apartamentos están abiertos muchos días al año pero solo se ocupan una parte pequeña de esos días.
- Se aprecia diferencia clara de ocupación por `room_type` y por barrio → ambas aportan poder predictivo.
- Varias columnas de reviews/scores tienen ~28 % nulos: habrá que decidir estrategia de imputación o excluir anuncios sin reviews.

---
## 2. Calendar — Disponibilidad diaria (variable objetivo principal)

Una fila por **listing × día**. La columna `available` es nuestra variable objetivo.

In [ ]:
# Carga con dtypes optimizados
calendar = pd.read_csv(
    RAW_DIR / 'calendar.csv',
    parse_dates=['date'],
    dtype={'available': 'category', 'listing_id': 'int64'},
    usecols=['listing_id','date','available','minimum_nights','maximum_nights']
)
print(f'Dimensiones: {calendar.shape[0]:,} filas × {calendar.shape[1]} columnas')
print(f'Memoria: {calendar.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print(f'Rango de fechas: {calendar["date"].min().date()} → {calendar["date"].max().date()}')
print(f'Listings únicos: {calendar["listing_id"].nunique():,}')
calendar.head()

In [ ]:
# Variable objetivo: occupied = 1 si está ocupado (available == 'f'), 0 si está libre
calendar['occupied'] = (calendar['available'] == 'f').astype(int)

print('Distribución de la variable objetivo:')
print(calendar['available'].value_counts())
print(f'\n% ocupado: {calendar["occupied"].mean()*100:.2f}%')
print(f'% disponible: {(1-calendar["occupied"].mean())*100:.2f}%')

# Plot distribución objetivo
fig, ax = plt.subplots(figsize=(6, 4))
calendar['available'].value_counts().plot(kind='bar', ax=ax, color=['steelblue','tomato'])
ax.set_title('Distribución de la variable objetivo (available)')
ax.set_xticklabels(['t = disponible','f = ocupado/bloqueado'], rotation=0)
ax.set_ylabel('Nº de filas')
plt.tight_layout()
plt.show()

In [ ]:
# Tasa de ocupación diaria (serie temporal)
daily = (calendar.groupby('date')['occupied'].mean() * 100).reset_index(name='pct_ocupado')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily['date'], daily['pct_ocupado'], color='steelblue', linewidth=1.2)
ax.set_title('Tasa de ocupación diaria (% listings ocupados por día)')
ax.set_ylabel('% ocupado')
ax.set_xlabel('Fecha')
plt.tight_layout()
plt.show()

print(f'% ocupación media: {daily["pct_ocupado"].mean():.1f}%')
print(f'Min: {daily["pct_ocupado"].min():.1f}% en {daily.loc[daily["pct_ocupado"].idxmin(),"date"].date()}')
print(f'Max: {daily["pct_ocupado"].max():.1f}% en {daily.loc[daily["pct_ocupado"].idxmax(),"date"].date()}')

In [ ]:
# Ocupación por día de la semana y por mes
calendar['dow'] = calendar['date'].dt.day_name()
calendar['month'] = calendar['date'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order_dow = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

dow_ocup = calendar.groupby('dow')['occupied'].mean().reindex(order_dow) * 100
sns.barplot(x=dow_ocup.index, y=dow_ocup.values, ax=axes[0], color='steelblue')
axes[0].set_title('% ocupado por día de la semana')
axes[0].set_ylabel('% ocupado')
axes[0].tick_params(axis='x', rotation=30)

month_ocup = calendar.groupby('month')['occupied'].mean() * 100
sns.barplot(x=month_ocup.index, y=month_ocup.values, ax=axes[1], color='steelblue')
axes[1].set_title('% ocupado por mes')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('% ocupado')

plt.tight_layout()
plt.show()

In [ ]:
# Ocupación cruzada con características del listing
cal_mini = calendar.merge(
    listings_mini[['id','neighbourhood_cleansed','room_type']].rename(columns={'id':'listing_id'}),
    on='listing_id', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

room_ocup = cal_mini.groupby('room_type')['occupied'].mean().sort_values(ascending=False) * 100
sns.barplot(x=room_ocup.values, y=room_ocup.index, ax=axes[0], color='steelblue')
axes[0].set_title('% ocupado por room_type')
axes[0].set_xlabel('% ocupado')

top_barrio_ocup = (cal_mini.groupby('neighbourhood_cleansed')['occupied']
                   .mean().sort_values(ascending=False).head(15) * 100)
sns.barplot(x=top_barrio_ocup.values, y=top_barrio_ocup.index, ax=axes[1], color='steelblue')
axes[1].set_title('% ocupado por barrio (top 15)')
axes[1].set_xlabel('% ocupado')

plt.tight_layout()
plt.show()

### Observaciones — Calendar

- ~6,6 M filas, sin nulos en `available`. Distribución equilibrada (~57 % disponibles, ~43 % ocupados).
- Rango real: **2025-12-14 → 2026-12-14** (scrape del 14/12/2025 cubre los 365 días siguientes).
- **Caveat importante:** `available == 'f'` en Inside Airbnb mezcla *reservado* y *bloqueado por el anfitrión*. En fechas cercanas (dic 25, ene 26) predomina la reserva; en fechas lejanas (jul-ago 26) muchos anfitriones aún no han abierto el calendario, así que aparecen como no disponibles sin estar realmente ocupados. Esto explica que el % 'ocupado' resulte **mayor en invierno (~55 %) que en verano (~35 %)** en la serie diaria — justo al revés de lo que dicta la demanda turística real.
- Leve patrón por día de la semana (viernes/sábado ~43 % vs lunes ~42 %) — diferencia pequeña por el mismo motivo anterior.
- Diferencias claras de ocupación **por barrio** y **por room_type** → son buenas predictoras.
- **Acción para el notebook 03:** acotar el horizonte temporal a las primeras **4-8 semanas** desde la fecha de scrape para que `occupied` refleje mayoritariamente ocupación real, no bloqueo.

---
## 3. Clima — Open-Meteo

In [ ]:
clima_path = RAW_DIR / 'clima_barcelona.csv'
if not clima_path.exists():
    print('⚠️  clima_barcelona.csv no existe todavía. Ejecuta antes 01_descarga_datos.ipynb.')
    clima = None
else:
    clima = pd.read_csv(clima_path, parse_dates=['date'])
    print(f'Dimensiones: {clima.shape[0]} filas × {clima.shape[1]} columnas')
    print(f'Rango de fechas: {clima["date"].min().date()} → {clima["date"].max().date()}')
    print(f'Nulos: {clima.isna().sum().sum()}')
    display(clima.describe().round(2))

In [ ]:
if clima is not None:
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

    axes[0].plot(clima['date'], clima['temperature_2m_max'], color='tomato', label='Máxima', alpha=0.8)
    axes[0].plot(clima['date'], clima['temperature_2m_min'], color='steelblue', label='Mínima', alpha=0.8)
    axes[0].fill_between(clima['date'], clima['temperature_2m_min'], clima['temperature_2m_max'],
                         color='lightgray', alpha=0.3)
    axes[0].set_title('Temperatura diaria (máx/mín) en Barcelona')
    axes[0].set_ylabel('°C')
    axes[0].legend()

    axes[1].bar(clima['date'], clima['precipitation_sum'], color='steelblue', width=1)
    axes[1].set_title('Precipitación diaria (mm)')
    axes[1].set_ylabel('mm')
    axes[1].set_xlabel('Fecha')

    plt.tight_layout()
    plt.show()

### Observaciones — Clima

- Serie limpia, sin nulos si el rango está dentro del histórico.
- Patrón estacional claro de temperatura.
- Precipitaciones concentradas en episodios puntuales, muchos días secos.
- `weathercode` es categórica (códigos WMO): en el notebook 03 la agruparemos en 4 categorías (soleado/nublado/lluvia/extremo).

---
## 4. Eventos — Ticketmaster

In [ ]:
eventos_path = RAW_DIR / 'eventos_barcelona.csv'
if not eventos_path.exists():
    print('⚠️  eventos_barcelona.csv no existe todavía. Ejecuta antes 01_descarga_datos.ipynb.')
    eventos = None
else:
    eventos = pd.read_csv(eventos_path, parse_dates=['date'])
    print(f'Dimensiones: {eventos.shape[0]} filas × {eventos.shape[1]} columnas')
    print(f'Rango de fechas: {eventos["date"].min().date()} → {eventos["date"].max().date()}')
    print(f'Duplicados por id: {eventos["id"].duplicated().sum()}')
    display(eventos.head())

In [ ]:
if eventos is not None:
    # Distribución por categoría
    cat_counts = eventos['category'].value_counts()
    print('Eventos por categoría:')
    print(cat_counts)

    fig, ax = plt.subplots(figsize=(10, 3.5))
    sns.barplot(x=cat_counts.values, y=cat_counts.index, ax=ax, color='steelblue')
    ax.set_title('Distribución de eventos por categoría')
    ax.set_xlabel('Número de eventos')
    plt.tight_layout()
    plt.show()

In [ ]:
if eventos is not None:
    # Volumen de eventos por día
    eventos_por_dia = eventos.groupby('date').size().reset_index(name='n_eventos')

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.bar(eventos_por_dia['date'], eventos_por_dia['n_eventos'], color='steelblue', width=1)
    ax.set_title('Número de eventos por día en Barcelona')
    ax.set_ylabel('Eventos/día')
    ax.set_xlabel('Fecha')
    plt.tight_layout()
    plt.show()

    # Solapamiento con el rango del calendar
    cal_range = (calendar['date'].min(), calendar['date'].max())
    overlap = eventos[(eventos['date'] >= cal_range[0]) & (eventos['date'] <= cal_range[1])]
    print(f'\nDías con eventos: {len(eventos_por_dia)}')
    print(f'Media eventos/día: {eventos_por_dia["n_eventos"].mean():.1f}')
    print(f'Máx eventos en un día: {eventos_por_dia["n_eventos"].max()}')
    print(f'Eventos dentro del rango del calendar: {len(overlap)} de {len(eventos)}')

### Observaciones — Eventos

- Ticketmaster solo sirve eventos futuros desde hoy, por lo que el rango cubierto es menor que el del calendar. Los días sin eventos se tratarán como `n_eventos = 0` en la integración.
- La variable clave para modelar será `n_eventos_por_dia`, no cada evento individual.
- Las categorías suelen estar dominadas por "Music" y "Miscellaneous". `price_min` / `price_max` vienen vacíos, no se usan.

---
## 5. Cruces preliminares

Visión rápida de la correlación temporal entre la **tasa de ocupación** y las dos señales externas (clima y eventos).

In [ ]:
if clima is not None and eventos is not None:
    # Unimos ocupación diaria con clima y eventos
    daily = daily.merge(clima, on='date', how='left')
    eventos_por_dia_full = (pd.date_range(daily['date'].min(), daily['date'].max())
                            .to_frame(index=False, name='date')
                            .merge(eventos_por_dia, on='date', how='left')
                            .fillna({'n_eventos':0}))
    daily = daily.merge(eventos_por_dia_full, on='date', how='left')

    # Correlaciones de Pearson contra pct_ocupado
    corr_cols = ['temperature_2m_max','temperature_2m_min','precipitation_sum','windspeed_10m_max','n_eventos']
    corr = daily[['pct_ocupado'] + corr_cols].corr().loc['pct_ocupado', corr_cols].round(3)
    print('Correlación de Pearson (diaria) vs % ocupado:')
    print(corr.to_string())
else:
    print('⚠️  Se omite el cruce porque faltan clima o eventos.')

In [ ]:
if clima is not None and eventos is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].scatter(daily['temperature_2m_max'], daily['pct_ocupado'], alpha=0.5, color='steelblue')
    axes[0].set_title('% ocupado vs temperatura máxima diaria')
    axes[0].set_xlabel('Temp. máx. (°C)')
    axes[0].set_ylabel('% ocupado')

    axes[1].scatter(daily['n_eventos'], daily['pct_ocupado'], alpha=0.5, color='darkorange')
    axes[1].set_title('% ocupado vs nº eventos/día')
    axes[1].set_xlabel('Nº eventos/día')
    axes[1].set_ylabel('% ocupado')

    plt.tight_layout()
    plt.show()

---
## 6. Conclusiones y siguiente paso

Hallazgos que condicionan el **notebook 03 (limpieza)**:

1. **Variable objetivo:** `occupied = (available == 'f')`, binaria, sin nulos, clases equilibradas (~43 % ocupado / ~57 % disponible).
2. **Caveat crítico:** `available='f'` mezcla reservas y bloqueos del anfitrión → limitar el horizonte del modelo a las primeras 4-8 semanas post-scrape.
3. **Listings:** quedarnos con ~25 columnas útiles, imputar nulos en reviews/scores (~28 %), descartar `price` (100 % nulo).
4. **Calendar:** convertir `available` a binaria `occupied`, extraer features temporales (día de semana, mes, temporada, fin de semana, festivo).
5. **Clima:** agrupar `weathercode` en categorías (soleado/nublado/lluvia/extremo); si hay nulos en fechas futuras, imputar con mismo día del año anterior.
6. **Eventos:** agregar a `n_eventos_por_dia`, rellenar con 0 los días sin eventos dentro del rango del calendar.

Después (**notebook 04 — integración**) uniremos los cuatro datasets en un único fichero `dataset_integrado.csv` con granularidad listing × día, listo para Orange Data Mining y Power BI.